# 环境配置

本节将完成 torchtitan-npu 的环境配置，确保你能在 Ascend NPU 上跑通 SFT 训练。

---

## 前置条件

在开始之前，确保以下软件已安装并可用：

| 组件 | 版本 | 验证命令 |
|------|------|--------|
| CANN | 9.0.0 | `npu-smi info` |
| PyTorch | 2.12.0 | `pip list \| grep torch"` |
| torch_npu | 2.12.0rc1 | `pip list \| grep torch_npu"` |

> **注意**：本教程假设 CANN 和 PyTorch 环境已由集群管理员配置好（`npu-smi info` 和 `import torch_npu` 均可正常执行）。如果不可用，请先参考 torchtitan-npu 官方的[安装指南](https://gitcode.com/cann/torchtitan-npu/blob/master/docs/user-guides/installation.md)完成 NPU 基础环境配置。

---

## 1. 克隆仓库

本教程使用 torchtitan-npu 的 最新分支。

```bash
git clone https://gitcode.com/cann/torchtitan-npu.git
cd torchtitan-npu
```

---

## 2. 安装依赖

```bash
pip install -r requirements.txt
pip install -e .
```

`requirements.txt` 中包含：

| 依赖 | 版本 | 用途 |
|------|------|------|
| `torch` | 2.12.0+cpu | PyTorch 基础框架 |
| `torch_npu` | 2.12.0rc1 | Ascend NPU 后端支持 |
| `torchtitan` | git (ac13e53) | 训练框架核心 |
| `triton-ascend` | 3.2.1 | Ascend Triton 加速库 |
| `safetensors` | 0.7.0 | 权重文件安全加载 |

---

## 3. 验证环境：跑 Debug 模型

torchtitan-npu 提供了 debug 模式，用极小模型快速验证环境是否正常：

```bash
NGPU=2 bash scripts/run_train.sh
```

Debug 模式的特点：
- 使用极小模型（hidden_size 很小，只有几层），几秒内完成一个 step。
- 不依赖提前下载的权重（随机初始化）。
- 只验证数据加载、模型构建、前向/反向传播、梯度更新是否正常。

**预期输出**：

```text
INFO:torchtitan_npu.entry:Starting training with config=sft_qwen3_1_7b_wordle_debug
INFO:torchtitan_npu.entry:Model built successfully
INFO:torchtitan_npu.entry:Step 1/10, loss=..., tps=...
...
INFO:torchtitan_npu.entry:Training completed
```

如果看到 `Training completed`，说明环境配置成功。

---

## 4. 下载模型权重

下载基座 Qwen3-1.7B 模型:

```bash
modelscope download --model PrimeIntellect/Qwen3-1.7B --local_dir ./assets/hf/Qwen3-1.7B
```

---

## 5. 下载数据集

Wordle SFT 训练使用 `willcb/V3-wordle` 数据集，包含 5 字母单词的猜词多轮对话：

```bash
# 1. 下载原始数据集
HF_ENDPOINT=https://hf-mirror.com download willcb/V3-wordle --repo-type=dataset --local-dir ./assets/data/wordle_raw

# 2. 转为 parquet 格式（torchtitan 要求）
python3 -c "
from datasets import load_dataset
ds = load_dataset('./assets/data/wordle_raw', split='train')
ds.to_parquet('./assets/data/wordle/train.parquet')
"
```

---

## 6. SFT 配置速览

在进入第 3 章之前，先看一眼我们即将使用的 SFT 配置入口：

```bash
# 正式训练命令（第 3 章会详细讲解）
NGPU=2 \
MODULE=torchtitan_npu.models.qwen3 \
CONFIG=sft_qwen3_1_7b_wordle \
bash scripts/run_train.sh
# ...
```

> **注意**：配置细节会在第 3 章数据准备和训练运行时详细展开，这里只需要知道"一条命令就能跑 SFT"。

---

## 小结

- Clone `mystri/torchtitan-npu` 的 `teaching-pipeline` 分支。
- `pip install -r requirements.txt && pip install -e .` 完成安装。
- 下载 Qwen3-1.7B-Wordle-SFT 权重。
- 用 debug 模式验证环境：`CONFIG=sft_qwen3_1_7b_wordle_debug bash scripts/run_train.sh`。

下一章，我们将真正启动 Wordle SFT 训练。

## 练习

1. （多选题）安装 torchtitan-npu 需要哪些步骤？
    A. 克隆仓库并切换到 teaching-pipeline 分支
    B. pip install -r requirements.txt && pip install -e .
    C. 编译 CANN 源码
    D. 下载 Qwen3-1.7B-Instruct 权重

2. （判断题）Debug 模式使用极小模型（随机初始化、几秒一个 step），目的是快速验证数据加载→模型构建→前向/反向→梯度更新全链路是否正常，不依赖下载好的权重。


In [ ]:
!cat ./answer/02.03_answer.txt